# LangGraph + A2A — Technical Support and Warranty Specialist

This notebook focuses on the **agent logic**.

- **Agent 1 — Technical Support:** a LangGraph workflow that decides whether warranty delegation is required.
- **Agent 2 — Warranty Specialist:** another LangGraph workflow that owns the `check_warranty` tool.
- **A2A:** Agent 1 communicates with the separately running Warranty Specialist server.

> LangGraph orchestrates each agent internally. A2A connects the independent agents.



## Architecture

```text
User
  |
  v
Technical Support LangGraph
  |
  +-- no warranty needed --> final answer
  |
  +-- warranty needed
          |
          | A2A
          v
    Warranty Specialist LangGraph
          |
          v
    check_warranty()
          |
          | A2A response
          v
Technical Support --> final answer
```

In [1]:
# Install once:
# %pip install -r requirements.txt

import re
import uuid
import httpx

from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from mistune import markdown

WARRANTY_AGENT_URL = "http://127.0.0.1:9999"

## 1. Warranty data and tool

In [2]:
WARRANTIES = {
    "LAP-123": {
        "product": "ProBook Laptop",
        "status": "active",
        "expires": "2027-12-15",
        "coverage": "hardware repair",
    },
    "LAP-456": {
        "product": "ProBook Laptop",
        "status": "expired",
        "expires": "2025-06-01",
        "coverage": "none",
    },
}

def check_warranty(serial_number: str) -> str:
    """Check warranty coverage for a device."""
    record = WARRANTIES.get(serial_number.upper())

    if not record:
        return f"No warranty record was found for {serial_number}."

    if record["status"] == "expired":
        return (
            f"{record['product']} ({serial_number}) is NOT covered. "
            f"The warranty expired on {record['expires']}."
        )

    return (
        f"{record['product']} ({serial_number}) is covered for "
        f"{record['coverage']} until {record['expires']}."
    )

## 2. Agent 2 — Warranty Specialist LangGraph

In [3]:
class WarrantyState(TypedDict):
    query: str
    serial_number: str | None
    result: str

def extract_serial(state: WarrantyState):
    # Added ?: to make the group non-capturing
    match = re.search(r"(?:PRINTER-|SCANNER-|LAP-)\d+", state["query"].upper())
    return {"serial_number": match.group(0) if match else None}

def verify_coverage(state: WarrantyState):
    serial = state["serial_number"]

    if not serial:
        return {"result": "Please provide a serial number such as LAP-123."}

    return {"result": check_warranty(serial)}

warranty_builder = StateGraph(WarrantyState)
warranty_builder.add_node("extract_serial", extract_serial)
warranty_builder.add_node("check_warranty", verify_coverage)

warranty_builder.add_edge(START, "extract_serial")
warranty_builder.add_edge("extract_serial", "check_warranty")
warranty_builder.add_edge("check_warranty", END)

warranty_graph = warranty_builder.compile()

In [4]:
# Test Agent 2 locally before introducing A2A.
result = warranty_graph.invoke({
    "query": "Is LAP-123 covered for hardware repair?",
    "serial_number": "",
    "result": "",
})

print(result["result"])

ProBook Laptop (LAP-123) is covered for hardware repair until 2027-12-15.


The same `warranty_graph` is hosted by `warranty_server.py`.

The Support Agent below will **not** invoke this local graph. It communicates with the remote copy through A2A.

## 3. Start the Warranty Specialist A2A server

In a separate terminal:

```bash
python warranty_server.py
```

Then inspect its Agent Card.

In [5]:
async with httpx.AsyncClient() as client:
    response = await client.get(
        f"{WARRANTY_AGENT_URL}/.well-known/agent-card.json"
    )
    response.raise_for_status()
    card = response.json()

print("Agent:", card["name"])
print("Description:", card["description"])
print("Skills:", [skill["name"] for skill in card["skills"]])

Agent: Warranty Specialist
Description: Verifies printer/device warranty coverage using the warranty provider API.
Skills: ['Check warranty coverage']


## 4. A2A delegation helper

In [6]:
async def ask_warranty_specialist(serial_number: str) -> str:
    """Delegate warranty verification to Agent 2 over A2A."""

    payload = {
        "jsonrpc": "2.0",
        "id": str(uuid.uuid4()),
        "method": "message/send",
        "params": {
            "message": {
                "role": "user",
                "messageId": str(uuid.uuid4()),
                "parts": [{
                    "kind": "text",
                    "text": (
                        f"Check whether {serial_number} is covered for a hardware repair."
                    ),
                }],
            }
        },
    }

    async with httpx.AsyncClient() as client:
        response = await client.post(WARRANTY_AGENT_URL, json=payload, timeout=30)
        response.raise_for_status()
        data = response.json()

    if "error" in data:
        raise RuntimeError(data["error"])

    result = data["result"]

    if result.get("kind") == "message":
        return result["parts"][0]["text"]

    if result.get("artifacts"):
        return result["artifacts"][-1]["parts"][0]["text"]

    message = result.get("status", {}).get("message")
    if message:
        return message["parts"][0]["text"]

    return str(result)

In [8]:
# Direct A2A test
print(await ask_warranty_specialist("PRINTER-1001"))

The device PRINTER-1001 is covered for a hardware repair.


## 5. Agent 1 — Technical Support LangGraph

The Support Agent has a routing decision:

```text
analyze_issue
      |
      v
warranty required?
   /       \
 NO        YES
 |          |
 v          v
answer    A2A delegation
            |
            v
        final answer
```

In [9]:
class SupportState(TypedDict):
    query: str
    diagnosis: str
    serial_number: str | None
    needs_warranty: bool
    warranty_result: str
    final_answer: str


def analyze_issue(state: SupportState):
    query = state["query"]
    match = re.search(r"(?:PRINTER-|SCANNER-|LAP-)\d+", query.upper())

    warranty_terms = (
        "warranty",
        "covered",
        "coverage",
        "insurance",
        "repair covered",
    )

    needs_warranty = any(
        term in query.lower()
        for term in warranty_terms
    )

    return {
        "diagnosis": (
            "The black screen may indicate a display or "
            "other hardware-related failure."
        ),
        "serial_number": match.group(0) if match else None,
        "needs_warranty": needs_warranty,
    }

def route_request(
    state: SupportState,
) -> Literal["delegate_warranty", "answer_directly"]:
    if state["needs_warranty"]:
        return "delegate_warranty"
    return "answer_directly"

async def delegate_warranty(state: SupportState):
    serial = state["serial_number"]

    if not serial:
        return {
            "warranty_result": (
                "A serial number is required to check coverage."
            )
        }

    result = await ask_warranty_specialist(serial)
    return {"warranty_result": result}

def answer_directly(state: SupportState):
    return {
        "final_answer": (
            f"Diagnosis:\\n{state['diagnosis']}\\n\\n"
            "Try connecting an external display and restarting "
            "the laptop. If the problem continues, the device "
            "may require hardware service."
        )
    }

def answer_with_warranty(state: SupportState):
    return {
        "final_answer": (
            f"Diagnosis:\\n{state['diagnosis']}\\n\\n"
            f"Warranty Specialist:\\n"
            f"{state['warranty_result']}\\n\\n"
            "Recommendation:\\n"
            "If the hardware failure is confirmed and covered, "
            "schedule a warranty repair."
        )
    }

In [10]:
support_builder = StateGraph(SupportState)

support_builder.add_node("analyze_issue", analyze_issue)
support_builder.add_node("delegate_warranty", delegate_warranty)
support_builder.add_node("answer_directly", answer_directly)
support_builder.add_node("answer_with_warranty", answer_with_warranty)

support_builder.add_edge(START, "analyze_issue")

support_builder.add_conditional_edges(
    "analyze_issue",
    route_request,
    {
        "delegate_warranty": "delegate_warranty",
        "answer_directly": "answer_directly",
    },
)

support_builder.add_edge(
    "delegate_warranty",
    "answer_with_warranty",
)

support_builder.add_edge("answer_directly", END)
support_builder.add_edge("answer_with_warranty", END)

support_graph = support_builder.compile()

## 6. Test route 1 — no A2A delegation

In [14]:
query = "My laptop screen is black. What should I try?"

result = await support_graph.ainvoke({
    "query": query,
    "diagnosis": "",
    "serial_number": None,
    "needs_warranty": False,
    "warranty_result": "",
    "final_answer": "",
})

print(result["final_answer"])

Diagnosis:\nThe black screen may indicate a display or other hardware-related failure.\n\nTry connecting an external display and restarting the laptop. If the problem continues, the device may require hardware service.


Because the user only asks for troubleshooting, the graph follows:

```text
analyze_issue → answer_directly
```

The Warranty Specialist is never contacted.

## 7. Test route 2 — delegate to Agent 2 over A2A

In [16]:
query = '''
My laptop screen is black.
The serial number is LAP-123.
Is the repair covered by the warranty?
'''

result = await support_graph.ainvoke({
    "query": query,
    "diagnosis": "",
    "serial_number": None,
    "needs_warranty": False,
    "warranty_result": "",
    "final_answer": "",
})

print(result["final_answer"])

Diagnosis:\nThe black screen may indicate a display or other hardware-related failure.\n\nWarranty Specialist:\nThe device LAP-123 is covered for a hardware repair. However, please note that coverage does not automatically qualify you for a free repair or replacement. A human review is required to determine the next steps.\n\nRecommendation:\nIf the hardware failure is confirmed and covered, schedule a warranty repair.


This time LangGraph chooses:

```text
analyze_issue
      |
      v
delegate_warranty
      |
      | A2A
      v
Warranty Specialist LangGraph
      |
      v
check_warranty
      |
      | A2A result
      v
answer_with_warranty
```

## 8. Key takeaway

There are **two different orchestration layers**:

### LangGraph
Controls what happens **inside an agent**.

The Technical Support Agent uses LangGraph to decide whether it can answer directly or needs specialist help.

The Warranty Specialist uses LangGraph to extract the device serial number and run its warranty check.

### A2A
Controls communication **between independent agents**.

Agent 1 does not import or call Agent 2's `check_warranty()` function. It sends an A2A request to the separately running Warranty Specialist.

That separation is the main concept of this example.